# Notebook 05 — Inference, Alerts & 72-Hour Forecast
## AeroTwinML · Multi-City Per-Horizon Forecasting

**Objective:** Load trained per-horizon models, generate forecasts for both cities,
check alert thresholds, and visualize the 72-hour forecast.

**Inference flow:**
1. Load latest data for each city
2. Build features
3. Load per-horizon models (24h, 48h, 72h)
4. Predict separately for each horizon
5. Generate alerts if AQI > 200
6. Embed weather + history in forecast JSON

In [ ]:
import sys, json
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from utils.config import get
from utils.storage import load_parquet, save_json
from feature_store.feature_builder import FeatureBuilder
from models.inference import InferenceEngine
from models.registry import get_models_by_horizon

plt.style.use('dark_background')
sns.set_palette('viridis')
%matplotlib inline

# Load data
DATA_DIR = Path(get('storage.data_dir', '../data'))
df = load_parquet(DATA_DIR / 'processed' / 'merged_hourly' / 'merged_latest.parquet')
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f'Loaded {len(df)} rows, cities: {df["city"].unique().tolist() if "city" in df.columns else ["single"]}')

## 1. Load Per-Horizon Models

In [ ]:
# Check available models
horizon_models = get_models_by_horizon()

if horizon_models:
    print('Per-horizon models loaded:')
    for h, entry in horizon_models.items():
        print(f'  {h}: {entry["model_name"]} (RMSE={entry.get("rmse", "N/A")})')
else:
    print('No per-horizon models found. Run notebook 03 first.')
    print('Falling back to single model...')

## 2. Generate Forecasts for Each City

In [ ]:
engine = InferenceEngine()

# Build features
builder = FeatureBuilder(df)
featured = builder.build_all()

cities = df['city'].unique().tolist() if 'city' in df.columns else [None]
city_forecasts = {}

for city_name in cities:
    if city_name:
        city_mask = featured['city'] == city_name if 'city' in featured.columns else pd.Series([True]*len(featured))
        city_featured = featured[city_mask]
        city_merged = df[df['city'] == city_name] if 'city' in df.columns else df
    else:
        city_featured = featured
        city_merged = df
    
    if city_featured.empty:
        continue
    
    forecast = engine.predict(city_featured)
    forecast['city'] = city_name
    city_forecasts[city_name or 'default'] = forecast
    
    print(f'\n=== {city_name} ===')
    print(f'  Current AQI: {forecast["current_aqi"]}')
    print(f'  Model info: {forecast["model_info"]}')
    for h in ['24h', '48h', '72h']:
        fc = forecast['forecast'][h]
        print(f'  {h}: AQI={fc["aqi"]}, category={fc["category"]}, alert={fc["alert"]}')

## 3. Forecast Comparison: Hyderabad vs Karachi

In [ ]:
if len(city_forecasts) > 1:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    horizons = ['24h', '48h', '72h']
    x = np.arange(len(horizons))
    width = 0.35
    
    for i, (city, fc) in enumerate(city_forecasts.items()):
        values = [fc['forecast'][h]['aqi'] for h in horizons]
        offset = width * (i - 0.5)
        bars = ax.bar(x + offset, values, width, label=city, alpha=0.8)
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f'{val:.0f}', ha='center', va='bottom', fontsize=10)
    
    ax.axhline(y=50, color='#00e400', linestyle='--', alpha=0.5, label='Good (50)')
    ax.axhline(y=100, color='#ffff00', linestyle='--', alpha=0.5, label='Moderate (100)')
    ax.axhline(y=200, color='#ff3333', linestyle='--', alpha=0.5, label='Alert (200)')
    
    ax.set_xlabel('Forecast Horizon')
    ax.set_ylabel('AQI')
    ax.set_title('72-Hour Forecast: Hyderabad vs Karachi')
    ax.set_xticks(x)
    ax.set_xticklabels(horizons)
    ax.legend()
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

## 4. Alert Check

In [ ]:
from utils.aqi_utils import is_alert_level, classify_aqi

ALERT_THRESHOLD = 200

alerts = []
for city, fc in city_forecasts.items():
    current = fc.get('current_aqi', 0)
    if is_alert_level(current):
        alerts.append({'city': city, 'type': 'current', 'aqi': current, 'category': classify_aqi(current).value})
    
    for h, data in fc.get('forecast', {}).items():
        if data.get('alert'):
            alerts.append({'city': city, 'type': f'forecast_{h}', 'aqi': data['aqi'], 'category': data['category']})

if alerts:
    print(f'ALERTS TRIGGERED ({len(alerts)}):')
    for a in alerts:
        print(f'  {a["city"]}: {a["type"]} AQI={a["aqi"]} ({a["category"]})')
else:
    print('No alerts triggered. All AQI values within safe levels.')
    for city, fc in city_forecasts.items():
        print(f'  {city}: current={fc["current_aqi"]}, max_forecast={max(fc["forecast"][h]["aqi"] for h in ["24h","48h","72h"])}')

## 5. Build and Save Forecast JSON

In [ ]:
# Build combined forecast JSON (same format as pipeline output)
if len(city_forecasts) == 1:
    forecast_json = list(city_forecasts.values())[0]
else:
    primary = list(city_forecasts.values())[0]
    forecast_json = {
        **primary,
        'cities': city_forecasts,
    }

# Save
output_path = DATA_DIR / 'processed' / 'predictions' / 'forecast_latest.json'
save_json(forecast_json, output_path)
print(f'Saved forecast to: {output_path}')
print(f'\nForecast JSON structure:')
print(json.dumps({k: v for k, v in forecast_json.items() if k != 'history'}, indent=2, default=str)[:2000])

## 6. Historical Forecast Visualization

In [ ]:
# Plot recent AQI history per city with forecast overlay
aqi_col = 'aqi' if 'aqi' in df.columns else 'om_forecast_aqi'

if aqi_col and 'city' in df.columns:
    fig, ax = plt.subplots(figsize=(16, 6))
    
    for city in df['city'].unique():
        city_df = df[df['city'] == city].sort_values('timestamp').tail(168)  # last 7 days
        ax.plot(city_df['timestamp'], city_df[aqi_col], label=f'{city} (historical)', linewidth=1)
        
        # Overlay forecast
        if city in city_forecasts:
            fc = city_forecasts[city]
            last_ts = city_df['timestamp'].iloc[-1]
            for i, h in enumerate([24, 48, 72]):
                fc_ts = last_ts + pd.Timedelta(hours=h)
                fc_aqi = fc['forecast'][f'{h}h']['aqi']
                ax.scatter(fc_ts, fc_aqi, s=100, zorder=5, marker='*')
                ax.annotate(f'{fc_aqi:.0f}', (fc_ts, fc_aqi), textcoords='offset points',
                           xytext=(0, 10), ha='center', fontsize=9)
    
    ax.axhline(y=100, color='#ffff00', linestyle='--', alpha=0.3)
    ax.axhline(y=200, color='#ff3333', linestyle='--', alpha=0.3)
    ax.set_title('7-Day History + 72-Hour Forecast')
    ax.set_xlabel('Time')
    ax.set_ylabel('AQI')
    ax.legend()
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

## Summary

**Complete pipeline demonstrated:**
1. Data ingestion from multi-city sources
2. Feature engineering with city encoding
3. Per-horizon model training (separate models for 24h/48h/72h)
4. SHAP explainability
5. Multi-city inference with differentiated forecasts
6. Alert system for hazardous AQI levels
7. Forecast JSON generation for dashboard consumption

**Architecture:**
```
Open-Meteo + OpenAQ (Hyderabad + Karachi)
       |
  FeatureBuilder (time + city + lag + weather + interaction)
       |
  Per-Horizon Training (ridge/RF/GB/XGB/LGB per 24h/48h/72h)
       |
  InferenceEngine (loads per-horizon models)
       |
  Forecast JSON (per-city, per-horizon)
       |
  Dashboard (city selector, charts, alerts)
```